# Prueba del módulo `limpieza.py`
Este notebook prueba las clases `dataloader` y `cleaner` del script `limpieza.py` utilizando un archivo Excel ubicado en `data/raw/`.

## 0. Configuración de rutas

In [ ]:
import sys
from pathlib import Path

# Agregar la carpeta 'src' al path para importar limpieza.py
ruta_src_datacleaning = Path('..') / 'src' / 'datacleaning'
sys.path.append(str(ruta_src_datacleaning))

# Ruta al archivo Excel de prueba
RUTA_ARCHIVO = Path('..') / 'data' / 'raw' / 'copia_calidad_docentes.xlsx'  # <- Ajustar al nombre real del archivo
COLUMNA_CLAVE = 'Si desea complementar sus respuestas o hacer sugerencias para nuestro mejoramiento continuo, por favor inclúyalas a continuación (si no tiene comentarios, dejar este espacio en blanco):'                                      # <- Ajustar al nombre real de la columna
HOJA = 0                                                     # <- Índice o nombre de la hoja Excel
RUTA_SALIDA = Path('..') / 'data' / 'processed' / 'datos_limpios_prueba1.csv'

print(f"Ruta archivo : {RUTA_ARCHIVO.resolve()}")
print(f"Archivo existe: {RUTA_ARCHIVO.exists()}")

## 1. Importar módulo

In [ ]:
# importar las clases desde el .py
from datacleaning.limpieza import Cleaner

print("Módulo importado correctamente.")

## 2. Prueba de `dataloader` — carga del archivo Excel

## 2.1. Prueba carga - sin recurrir a la clase

> Verificar cómo reconoce Pandas la columna clave

In [ ]:
import pandas as pd

df_prueba1 = pd.read_excel(RUTA_ARCHIVO, sheet_name=HOJA)
df_prueba1.info()  # Mostrar información del DataFrame antes de la limpieza

In [ ]:
df_prueba1.columns

In [ ]:
print(f"DEBUG: El dtype es: {df_prueba1[COLUMNA_CLAVE].dtype}")
print(f"DEBUG: Los primeros 5 valores son: {df_prueba1[COLUMNA_CLAVE].head().tolist()}")

## 2.2. Prueba de carga con la clase `Cleaner`

In [ ]:
cleaner = Cleaner(
    file_path=RUTA_ARCHIVO,
    survey_name="stopwords_calidaddocentes", #Calidad Docentes
    key_column=COLUMNA_CLAVE,
    sheet_name=HOJA
)

df = cleaner.load_data()

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print(f"Columnas disponibles: {df.columns.tolist()}")
df.head()

## 3. Prueba de `cleaner.limpiar_texto` — limpieza unitaria

In [ ]:
# Casos de prueba individuales
casos = [
    "  ¡Hola Mundo!  ",
    "Ángela estudió en la Universidad Académica",
    "Texto con: caracteres@especiales #raros!",
    "España y la ñ deben conservarse",
]

print(f"{'Texto original':<45} -> {'Texto limpio'}")
print("-" * 75)
for caso in casos:
    print(f"{repr(caso):<45} -> {repr(cleaner._clean_text(caso))}")

## 4. Prueba de `cleaner.limpiar_columna_clave` — limpieza sobre el DataFrame

In [ ]:
df_limpio = cleaner.clean_key_column()

print(f"Columna nueva generada: '{COLUMNA_CLAVE}_clean'")
df_limpio[[COLUMNA_CLAVE, f"{COLUMNA_CLAVE}_clean"]].head(10)

## 5. Prueba de `cleaner.save_cleaned_data` — exportar a CSV

In [ ]:
# Crear carpeta de salida si no existe
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)

cleaner.save_cleaned_data(out_path=str(RUTA_SALIDA))

print(f"Archivo guardado en: {RUTA_SALIDA.resolve()}")

In [ ]:
import pandas as pd
datos_limpios = pd.read_csv(RUTA_SALIDA)

In [ ]:

datos_limpios.dropna(how='all').iloc[:, -2:].sort_values(by=datos_limpios.columns[-1])  # Mostrar las dos últimas columnas

## 6. Prueba de manejo de errores

In [ ]:
# 6.1 Archivo inexistente
print("--- Archivo inexistente ---")
try:
    loader_err = Cleaner(file_path='no_existe.xlsx', key_column=COLUMNA_CLAVE,
                         survey_name="stopwords_calidaddocentes", sheet_name=HOJA)
    loader_err.load_data()
except (FileNotFoundError, ValueError) as e:
    print(f"[OK] Error capturado: {e}")

In [ ]:
# 6.2 Columna clave inexistente
print("\n--- Columna clave inexistente ---")
try:
    loader_err = Cleaner(file_path=RUTA_ARCHIVO, key_column='columna_falsa',
                    survey_name="stopwords_calidaddocentes", sheet_name=HOJA)

    loader_err.load_data()
except ValueError as e:
    print(f"[OK] Error capturado: {e}")

In [ ]:
# 6.3 Formato no soportado
print("\n--- Formato no soportado ---")
try:
    loader_err = Cleaner(file_path='archivo.txt', key_column=COLUMNA_CLAVE,
                    survey_name="stopwords_calidaddocentes", sheet_name=HOJA)

    loader_err.load_data()
except (FileNotFoundError, ValueError) as e:
    print(f"[OK] Error capturado: {e}")

# 7. Prueba del módulo `synonyms.py`
Este notebook prueba la clase `SynonymReplacer` del módulo `synonyms.py`, encargada de reemplazar términos por su sinónimo más común usando un modelo Word2Vec preentrenado en español (SBWCE) combinado con coloquialismos colombianos añadidos manualmente.

## 7.0. Configuración e importación

In [ ]:
# importar la clase desde el .py (misma carpeta que datacleaning.limpieza)
from datacleaning.synonyms import SynonymReplacer

print("Módulo importado correctamente.")

## 7.1. Prueba de `load_model` — carga (y descarga si aplica) del modelo Word2Vec

In [ ]:
# NOTA: si el modelo no existe en SYNONYMS_MODEL_PATH, esta celda lo
# descargará automáticamente (~2GB), por lo que puede tardar varios
# minutos la primera vez que se ejecute.
replacer = SynonymReplacer()

replacer.load_model()

print(f"Modelo cargado correctamente.")
print(f"Tamaño del vocabulario: {len(replacer.model.key_to_index)}")

## 7.2. Prueba de `build_synonyms_dict` — autogeneración del diccionario de sinónimos

In [ ]:
synonyms_dict = replacer.build_synonyms_dict()

print(f"Total de términos mapeados: {len(synonyms_dict)}")
print("Ejemplo de entradas generadas:")
for palabra, canonico in list(synonyms_dict.items())[:15]:
    print(f"  {palabra:<15} -> {canonico}")

In [ ]:
# Verificación puntual: los coloquialismos colombianos deben estar
# presentes y apuntar al canónico esperado
for coloquial, esperado in replacer.colombian_colloquialisms.items():
    obtenido = synonyms_dict.get(coloquial)
    estado = "[OK]" if obtenido == esperado else "[FALLO]"
    print(f"{estado} '{coloquial}' -> '{obtenido}' (esperado: '{esperado}')")

## 7.3. Prueba de `replace_synonyms` — casos unitarios

In [ ]:
# Casos de prueba individuales
casos_sinonimos = [
    "el servicio fue maravilloso",
    "la comida estuvo terrible",
    "todo estuvo bacano y muy chevere",
    "EXCELENTE atencion, MUY BUENA",              # mayusculas
    "el profesor uso xilofono en la clase",         # palabra fuera de vocabulario
    "muy chimba la dinamica de la materia",
]

print(f"{'Texto original':<45} -> {'Texto con sinonimos reemplazados'}")
print("-" * 90)
for caso in casos_sinonimos:
    print(f"{repr(caso):<45} -> {repr(replacer.replace_synonyms(caso))}")

In [ ]:
# Caso de valor no-string (ej. NaN representado como float)
print(f"Entrada no-str (None)  -> {repr(replacer.replace_synonyms(None))}")
print(f"Entrada no-str (float) -> {repr(replacer.replace_synonyms(3.1416))}")

## 7.4. Prueba de `synonyms_to_dataframe` — aplicación sobre el DataFrame limpio
Se reutiliza el `df_limpio` generado en la sección 4 con `cleaner.clean_key_column()`.

In [ ]:
df_con_sinonimos = replacer.synonyms_to_dataframe(
    data=datos_limpios,
    column=f"{COLUMNA_CLAVE}_clean"
)

print(f"Columna nueva generada: '{COLUMNA_CLAVE}_clean_synonyms'")
df_con_sinonimos[[f"{COLUMNA_CLAVE}_clean", f"{COLUMNA_CLAVE}_clean_synonyms"]].head(10)

## 7.5. Prueba de manejo de errores

In [ ]:
# 7.5.1 Columna inexistente
print("--- Columna inexistente ---")
try:
    replacer.synonyms_to_dataframe(df_limpio, column='columna_falsa')
except ValueError as e:
    print(f"[OK] Error capturado: {e}")

In [ ]:
# 7.5.2 build_synonyms_dict sin haber cargado el modelo
print("\n--- build_synonyms_dict sin load_model ---")
try:
    replacer_sin_modelo = SynonymReplacer()
    replacer_sin_modelo.build_synonyms_dict()
except ValueError as e:
    print(f"[OK] Error capturado: {e}")

In [ ]:
# 7.5.3 replace_synonyms sin haber construido el diccionario
print("\n--- replace_synonyms sin build_synonyms_dict ---")
try:
    replacer_sin_dict = SynonymReplacer()
    replacer_sin_dict.replace_synonyms("un texto cualquiera")
except ValueError as e:
    print(f"[OK] Error capturado: {e}")

In [ ]:
# 7.5.4 synonyms_to_dataframe sin haber construido el diccionario
print("\n--- synonyms_to_dataframe sin build_synonyms_dict ---")
try:
    replacer_sin_dict2 = SynonymReplacer()
    replacer_sin_dict2.synonyms_to_dataframe(df_limpio, column=f"{COLUMNA_CLAVE}_clean")
except ValueError as e:
    print(f"[OK] Error capturado: {e}")